# C7 核心检索 benchmark（仅 BM25/CCH）

本页把改动前后比较改成一个可重复的统一 benchmark：从 canonical `queries/qrels` 固定选取同一批同时标为 `demo` 和 `regression` 的问题；BM25 基线与已有的可靠章节标题（CCH）检索增强共享同一证据语料、`top_k` 和字符预算。

检索阶段只接收问题文字和可逐字定位的 canonical evidence 原文。所有方法完成排序后才读取 qrels 计算指标，因此参考答案、目标页和 qrels 不会作为排序特征。这里使用当前 canonical 数据包中的全部 evidence；它仍是围绕教程问题整理的标注原文池，不是完整 PDF 分块库，所以结果只比较该池内的排序变化，不能排除语料构造偏差，也不能外推为全书召回率。每道题保留首条相关证据排名、coverage、Recall@k、MRR、相对 BM25 的改善/不变/退化（若覆盖与排名方向相反则记录 `tradeoff`），并给出方法汇总。缺少 canonical 数据或必要依赖会直接失败，不使用 fallback。

## 本页范围

这是核心检索 benchmark，不是端到端回答评测：统一脚本当前只比较 BM25 与 CCH 两种检索排序。其余专题 Notebook 的结果属于各自的案例审计；要验收完整的“问题 → 检索/补查 → 后处理 → 回答 → 引用或拒答”，请执行[端到端验收协议](端到端验收协议.md)。

In [1]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path


def find_course_root(start: Path) -> Path:
    for folder in (start.resolve(), *start.resolve().parents):
        candidate = folder / "notebook" / "C7 高级 RAG 技巧"
        if (folder / "data" / "dataset" / "manifest.json").is_file() and (folder / "common" / "dataset.py").is_file():
            return folder
        if (candidate / "data" / "dataset" / "manifest.json").is_file() and (candidate / "common" / "dataset.py").is_file():
            return candidate
    raise FileNotFoundError("找不到 C7 canonical dataset")


course_root = find_course_root(Path.cwd())
script_path = course_root / "scripts" / "run_benchmark.py"
if not script_path.is_file():
    raise FileNotFoundError(f"横评入口不存在：{script_path}")
spec = importlib.util.spec_from_file_location("c7_run_benchmark_notebook", script_path)
if spec is None or spec.loader is None:
    raise ImportError(f"无法加载横评入口：{script_path}")
benchmark = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = benchmark
spec.loader.exec_module(benchmark)

report = benchmark.run_benchmark()
print(benchmark.render_text(report))

C7 同题统一横评
题数=6；方法=bm25,bm25_cch；top_k=4；char_budget=1200

逐题结果（rank / coverage / Recall@k / MRR / 相对 BM25）
- model_selection_with_intro_scope: 《南瓜书》绪论里说，机器学习算法之间有没有绝对更好的一个？
  bm25: rank=1; coverage=1.000; Recall@4=1.000; MRR=1.000; outcome=baseline
  bm25_cch: rank=1; coverage=1.000; Recall@4=1.000; MRR=1.000; outcome=unchanged
- cross_validation_reliability: 交叉验证法为什么比单次留出法更可靠？
  bm25: rank=2; coverage=1.000; Recall@4=1.000; MRR=0.500; outcome=baseline
  bm25_cch: rank=2; coverage=0.500; Recall@4=0.500; MRR=0.500; outcome=degraded
- ensemble_learning_definition: 集成多个弱学习器来提升整体预测效果的方法是什么
  bm25: rank=4; coverage=1.000; Recall@4=1.000; MRR=0.250; outcome=baseline
  bm25_cch: rank=miss; coverage=0.000; Recall@4=0.000; MRR=0.000; outcome=degraded
- lda_recursive_derivation: LDA 从投影分离目标怎样推到 N−1 个最大广义特征值及其特征向量？请给出中间优化关系。
  bm25: rank=miss; coverage=0.000; Recall@4=0.000; MRR=0.000; outcome=baseline
  bm25_cch: rank=miss; coverage=0.000; Recall@4=0.000; MRR=0.000; outcome=unchanged
- newton_me

## 保存结构化结果与验证

下面的 JSON 是本次实际执行产生的完整逐题输出；`answer_context_source` 明确要求回答上下文只来自 canonical evidence 原文。

In [2]:
import json

benchmark.validate_report(report)
assert report["validation"]["same_query_batch_for_all_methods"]
assert report["validation"]["same_top_k_for_all_methods"]
assert report["validation"]["same_char_budget_for_all_methods"]
assert report["validation"]["qrels_used_for_metrics"]
assert not report["validation"]["target_pages_or_reference_answers_used_for_retrieval"]
assert report["validation"]["fallback_used"] is False
print("benchmark contract passed")
print(json.dumps(report, ensure_ascii=False, indent=2))

benchmark contract passed
{
  "schema_version": 1,
  "benchmark": "c7_same_question_retrieval",
  "config": {
    "query_ids": [
      "model_selection_with_intro_scope",
      "cross_validation_reliability",
      "ensemble_learning_definition",
      "lda_recursive_derivation",
      "newton_methods_comparison",
      "model_evaluation_followup"
    ],
    "query_scope": "explicit canonical demo/regression set",
    "top_k": 4,
    "char_budget": 1200,
    "methods": [
      "bm25",
      "bm25_cch"
    ],
    "baseline": "bm25",
    "qrels": "canonical qrels relevance=1, loaded after retrieval",
    "answer_context_source": "canonical evidence.quote only",
    "evidence_pool_scope": "curated exact-PDF evidence pool, not the full PDF chunk corpus"
  },
  "questions": {
    "model_selection_with_intro_scope": {
      "query": "《南瓜书》绪论里说，机器学习算法之间有没有绝对更好的一个？",
      "relevant_evidence_count": 1,
      "methods": {
        "bm25": {
          "retrieved_evidence_ids": [
            "evi_

## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[执行端到端验收](端到端验收.ipynb)

